# 06 — Remaining Time Prediction

Regression: given a prefix of k events, predict remaining days until case close.
Same methodology as NB04/NB05: chronological 60/20/20 split, leaky removal, Boolean+Succession encoding sweep, XGB/LGBM + LSTM + Transformer.

## 1. Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import warnings, os, random
import pm4py as pm
warnings.filterwarnings('ignore')
os.makedirs('figs', exist_ok=True)
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
random.seed(RANDOM_STATE)

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import MinMaxScaler
from xgboost import XGBRegressor
import lightgbm as lgb
from lightgbm import LGBMRegressor

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import math

## 2. Load & Clean

In [ ]:
dtype_dict = {
    'hired':'Int8','Rejected':'Int8','CW':'Int8','Evergreen':'Int8',
    'Region':'Int16','Country':'Int16','Job Family':'Int16','Job Family Group':'Int16',
}
df = pd.read_csv('data/'+'event_log_consolidated.csv', low_memory=False, dtype=dtype_dict, parse_dates=['timestamp'])
df = df[~df['Step'].str.match(r'^\d+$', na=False)].copy()
df.sort_values(['Case_id','timestamp'], inplace=True)
df.reset_index(drop=True, inplace=True)
print(f'Shape: {df.shape}  |  Cases: {df["Case_id"].nunique():,}')

Shape: (1133314, 17)  |  Cases: 279,458


## 3. Compute Case End Times + Format + Split

In [ ]:
# ── Strict chronological 60/20/20 split (leakage fix) ────────────────────────
case_times = df.groupby('case:concept:name').agg(
    case_start=('time:timestamp', 'min'),
    case_end  =('time:timestamp', 'max'),
).sort_values('case_start')

# case_end_times used throughout notebook as the regression target reference
case_end_times = case_times['case_end']

n            = len(case_times)
train_cutoff = case_times.iloc[int(n * 0.60)]['case_start']
val_cutoff   = case_times.iloc[int(n * 0.80)]['case_start']

train_ids = case_times[case_times['case_end']   <  train_cutoff].index
val_ids   = case_times[(case_times['case_start'] >= train_cutoff) &
                        (case_times['case_end']   <  val_cutoff)].index
test_ids  = case_times[case_times['case_start'] >= val_cutoff].index

train_df = df[df['case:concept:name'].isin(train_ids)].copy()

straddle_tv = case_times[(case_times['case_start'] <  train_cutoff) &
                          (case_times['case_end']   >= train_cutoff) &
                          (case_times['case_end']   <  val_cutoff)].index
val_df = pd.concat([
    df[df['case:concept:name'].isin(val_ids)],
    df[df['case:concept:name'].isin(straddle_tv) & (df['time:timestamp'] >= train_cutoff)],
]).sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

straddle_vt = case_times[(case_times['case_start'] <  val_cutoff) &
                          (case_times['case_end']   >= val_cutoff)].index
test_df = pd.concat([
    df[df['case:concept:name'].isin(test_ids)],
    df[df['case:concept:name'].isin(straddle_vt) & (df['time:timestamp'] >= val_cutoff)],
]).sort_values(['case:concept:name', 'time:timestamp']).reset_index(drop=True)

print(f'Train cutoff: {train_cutoff.date()} | Val cutoff: {val_cutoff.date()}')
print(f'Train — {train_df["case:concept:name"].nunique():,} cases  |  {len(train_df):,} events')
print(f'Val   — {val_df["case:concept:name"].nunique():,} cases  |  {len(val_df):,} events')
print(f'Test  — {test_df["case:concept:name"].nunique():,} cases  |  {len(test_df):,} events')


## 4. Leaky Activity Removal + Vocabulary

In [ ]:
LEAKY_THRESHOLD = 0.85
act_hire_rate = train_df.groupby('concept:name')['hired'].apply(lambda x: x.astype(float).mean())
leaky_acts = set(act_hire_rate[act_hire_rate >= LEAKY_THRESHOLD].index)
print(f'Leaky activities removed ({len(leaky_acts)}):')
for act in sorted(leaky_acts, key=lambda a: -act_hire_rate[a]):
    print(f'  {act_hire_rate[act]:.3f}  {act}')

train_df = train_df[~train_df['concept:name'].isin(leaky_acts)].copy()
val_df   = val_df[~val_df['concept:name'].isin(leaky_acts)].copy()
test_df  = test_df[~test_df['concept:name'].isin(leaky_acts)].copy()

all_train_acts = sorted(train_df['concept:name'].dropna().unique())
act2idx = {act: i+2 for i, act in enumerate(all_train_acts)}
act2idx['<PAD>'] = 0; act2idx['<UNK>'] = 1
VOCAB_SIZE = len(act2idx); MAX_SEQ_LEN = 30
print(f'Vocabulary: {VOCAB_SIZE} | MAX_SEQ_LEN: {MAX_SEQ_LEN}')

Leaky activities removed (17):
  1.000  Ready for Hire – Do not send Manager Survey
  1.000  Ready for Hire - Send Hiring Manager Survey
  0.997  Additional Offer Packet Document(s)
  0.990  Review Company and Office Guides
  0.986  Candidate Experience Survey
  0.984  Recruiting Process Survey
  0.976  Automatically Unpost Jobs
  0.968  Confirm no manager survey to be sent
  0.912  Colombia Standard Questionnaire (2024)
  0.909  Additional Background / Reference Check
  0.909  LATAM Standard Questionnaire (2024)
  0.900  Confirm no manager/additional hiring manager survey to be sent
  0.889  Renegotiate Offer
  0.885  Make Background Check Decision
  0.885  BGC / Examenes Preocupacionales
  0.884  Set Background Check Status
  0.853  Consolidated Approval by Initiator
Vocabulary: 39 | MAX_SEQ_LEN: 30


## 5. Encoding Helpers + Feature Extraction

In [ ]:
PM_COLS = ['case:concept:name','concept:name','time:timestamp']
SKIP_COLS = {'case:concept:name','concept:name','time:timestamp','@@diff_start_end'}
CASE_ATTR_COLS = ['Region','Country','Job Family','Job Family Group','CW','Evergreen']

case_attrs_train = train_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_val   = val_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)
case_attrs_test  = test_df.groupby('case:concept:name')[CASE_ATTR_COLS].first().fillna(0).astype(float)

def get_prefix_pm(df_clean, k):
    if k == 'all': return df_clean[PM_COLS].copy()
    return pm.get_prefixes_from_log(df_clean, length=k, case_id_key='case:concept:name')[PM_COLS].copy()

def bool_encoding(prefix_pm):
    enriched = pm.extract_outcome_enriched_dataframe(prefix_pm, activity_key='concept:name',
                    timestamp_key='time:timestamp', case_id_key='case:concept:name')
    feat_cols = [c for c in enriched.columns if c not in SKIP_COLS]
    return enriched.drop_duplicates('case:concept:name').set_index('case:concept:name')[feat_cols].sort_index()

def succ_encoding(prefix_pm):
    all_case_ids = sorted(prefix_pm['case:concept:name'].unique())
    df_s = prefix_pm.copy()
    df_s['prev_act'] = df_s.groupby('case:concept:name')['concept:name'].shift(1)
    df_s = df_s.dropna(subset=['prev_act'])
    if df_s.empty:
        return pd.DataFrame(index=pd.Index(all_case_ids, name='case:concept:name'))
    df_s['bigram'] = df_s['prev_act'] + ' >> ' + df_s['concept:name']
    return df_s.groupby(['case:concept:name','bigram']).size().unstack(fill_value=0).reindex(all_case_ids, fill_value=0).rename_axis('case:concept:name')

def extract_remaining_time(df_subset, k, case_end_times):
    records = {}
    for cid, grp in df_subset.groupby('case:concept:name', sort=False):
        grp = grp.sort_values('time:timestamp')
        prefix = grp if k == 'all' else grp.head(k)
        last_ts = prefix['time:timestamp'].max()
        end_ts  = case_end_times.get(cid)
        if end_ts is None: continue
        remaining = (end_ts - last_ts).total_seconds() / 86400
        records[cid] = max(0.0, remaining)
    return pd.Series(records, dtype=np.float32)

def build_Xy_reg(X_pm, case_attrs_split, y_series, train_cols=None):
    if train_cols is not None:
        X_pm = X_pm.reindex(columns=train_cols, fill_value=0)
    X = X_pm.join(case_attrs_split, how='left').fillna(0)
    common = X.index.intersection(y_series.index)
    X = X.loc[common]
    y = y_series.reindex(common).values.astype(np.float32)
    return X.values.astype(np.float32), y, list(X.columns)

def build_sequences(df_subset, case_ids, k, act2idx, max_seq_len):
    seqs = []
    grp_map = {cid: grp.sort_values('time:timestamp') for cid, grp in df_subset.groupby('case:concept:name', sort=False)}
    for cid in case_ids:
        grp  = grp_map.get(cid)
        acts = (grp['concept:name'].tolist() if grp is not None else [])
        pref = acts if k == 'all' else acts[:k]
        idxs = [act2idx.get(a, 1) for a in pref]
        seq  = idxs[-max_seq_len:]
        seqs.append([0] * (max_seq_len - len(seq)) + seq)
    return np.array(seqs, dtype=np.int64)

print('Helpers defined.')

Helpers defined.


## 6. Prefix Sweep — XGB / LGBM Regressors

In [ ]:
PREFIX_LENGTHS = [1, 3, 5, 10, 20, 'all']
ENCODINGS     = [('Boolean', bool_encoding), ('Succession', succ_encoding)]
results = {}

for enc_name, enc_fn in ENCODINGS:
    print(f'\n── {enc_name} ──────────────────────')
    for k in PREFIX_LENGTHS:
        y_tr_s = extract_remaining_time(train_df, k, case_end_times)
        y_va_s = extract_remaining_time(val_df,   k, case_end_times)
        y_te_s = extract_remaining_time(test_df,  k, case_end_times)

        X_tr_pm = enc_fn(get_prefix_pm(train_df, k)); tc = list(X_tr_pm.columns)
        X_va_pm = enc_fn(get_prefix_pm(val_df,   k))
        X_te_pm = enc_fn(get_prefix_pm(test_df,  k))

        X_tr, y_tr, _ = build_Xy_reg(X_tr_pm, case_attrs_train, y_tr_s)
        X_va, y_va, _ = build_Xy_reg(X_va_pm, case_attrs_val,   y_va_s, tc)
        X_te, y_te, _ = build_Xy_reg(X_te_pm, case_attrs_test,  y_te_s, tc)
        if len(X_tr) == 0 or len(X_te) == 0: continue

        for mname, reg in [
            ('XGB',  XGBRegressor(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE, verbosity=0,
                                  early_stopping_rounds=20, eval_metric='mae')),
            ('LGBM', LGBMRegressor(n_estimators=300, n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)),
        ]:
            if mname == 'XGB':
                reg.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                reg.fit(X_tr, y_tr, eval_set=[(X_va, y_va)],
                        callbacks=[lgb.early_stopping(20, verbose=False), lgb.log_evaluation(period=-1)])
            pred  = np.clip(reg.predict(X_te), 0, None)
            mae   = mean_absolute_error(y_te, pred)
            rmse  = np.sqrt(mean_squared_error(y_te, pred))
            r2    = r2_score(y_te, pred)
            medae = float(np.median(np.abs(y_te - pred)))
            results.setdefault(f'{enc_name}/{mname}', {})[str(k)] = {'MAE': mae, 'RMSE': rmse, 'R2': r2, 'MedAE': medae}
            print(f'  {mname:5s} k={str(k):4s}  MAE={mae:.2f}d  RMSE={rmse:.2f}d  R2={r2:.4f}')


── Boolean ──────────────────────
  XGB   k=1     MAE=20.33d  RMSE=21.77d  R2=-0.8292
  LGBM  k=1     MAE=22.20d  RMSE=23.44d  R2=-1.1210


KeyboardInterrupt: 

## 7. LSTM Regressor

In [ ]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

y_tr_full = extract_remaining_time(train_df, 'all', case_end_times)
y_va_full = extract_remaining_time(val_df,   'all', case_end_times)
y_te_full = extract_remaining_time(test_df,  'all', case_end_times)

X_tr_tab, _, _ = build_Xy_reg(bool_encoding(get_prefix_pm(train_df,'all')), case_attrs_train, y_tr_full)
X_va_tab, _, _ = build_Xy_reg(bool_encoding(get_prefix_pm(val_df,  'all')), case_attrs_val,   y_va_full, list(bool_encoding(get_prefix_pm(train_df,'all')).columns))
X_te_tab, _, _ = build_Xy_reg(bool_encoding(get_prefix_pm(test_df, 'all')), case_attrs_test,  y_te_full, list(bool_encoding(get_prefix_pm(train_df,'all')).columns))

scaler = MinMaxScaler()
X_tr_tab = scaler.fit_transform(X_tr_tab).astype(np.float32)
X_va_tab = scaler.transform(X_va_tab).astype(np.float32)
X_te_tab = scaler.transform(X_te_tab).astype(np.float32)
N_TAB = X_tr_tab.shape[1]

seqs_tr = build_sequences(train_df, y_tr_full.index, 'all', act2idx, MAX_SEQ_LEN)
seqs_va = build_sequences(val_df,   y_va_full.index, 'all', act2idx, MAX_SEQ_LEN)
seqs_te = build_sequences(test_df,  y_te_full.index, 'all', act2idx, MAX_SEQ_LEN)

class RegDataset(Dataset):
    def __init__(self, X, seqs, y):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.seqs = torch.tensor(seqs, dtype=torch.long)
        self.y = torch.tensor(y, dtype=torch.float32)
    def __len__(self): return len(self.y)
    def __getitem__(self, i): return self.X[i], self.seqs[i], self.y[i]

train_loader = DataLoader(RegDataset(X_tr_tab, seqs_tr, y_tr_full.values), batch_size=256, shuffle=True,  num_workers=0)
val_loader   = DataLoader(RegDataset(X_va_tab, seqs_va, y_va_full.values), batch_size=512, shuffle=False, num_workers=0)
test_loader  = DataLoader(RegDataset(X_te_tab, seqs_te, y_te_full.values), batch_size=512, shuffle=False, num_workers=0)
print(f'Train: {len(train_loader.dataset):,}  Val: {len(val_loader.dataset):,}  Test: {len(test_loader.dataset):,}')

Device: cuda
Train: 167,673  Val: 55,892  Test: 55,892


In [ ]:
class LSTMReg(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, hidden=128, n_layers=2, dropout=0.3, n_tab=10):
        super().__init__()
        self.emb  = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.lstm = nn.LSTM(emb_dim, hidden, num_layers=n_layers, batch_first=True,
                            dropout=dropout if n_layers > 1 else 0)
        self.fc1  = nn.Linear(hidden + n_tab, 64)
        self.fc2  = nn.Linear(64, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, seq, tab):
        _, (h, _) = self.lstm(self.emb(seq))
        return self.fc2(torch.relu(self.fc1(self.drop(torch.cat([h[-1], tab], 1))))).squeeze(1)


def train_reg_model(model, train_loader, val_loader, n_epochs=20):
    criterion = nn.MSELoss()
    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3, factor=0.5)
    best_mae, best_state, no_improve = float('inf'), None, 0
    for epoch in range(n_epochs):
        model.train()
        for X_b, seq_b, y_b in train_loader:
            X_b, seq_b, y_b = X_b.to(device), seq_b.to(device), y_b.to(device)
            optimizer.zero_grad()
            criterion(model(seq_b, X_b), y_b).backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        model.eval()
        pv, tv = [], []
        with torch.no_grad():
            for X_b, seq_b, y_b in val_loader:
                pv.extend(np.clip(model(seq_b.to(device), X_b.to(device)).cpu().numpy(), 0, None))
                tv.extend(y_b.numpy())
        val_mae = mean_absolute_error(tv, pv)
        scheduler.step(val_mae)
        print(f'  Epoch {epoch+1:02d}  val_MAE={val_mae:.3f}d')
        if val_mae < best_mae:
            best_mae = val_mae; best_state = {k: v.clone() for k, v in model.state_dict().items()}; no_improve = 0
        else:
            no_improve += 1
            if no_improve >= 5: print(f'  Early stop at epoch {epoch+1}'); break
    model.load_state_dict(best_state)
    return model

lstm_model = LSTMReg(VOCAB_SIZE, n_tab=N_TAB).to(device)
print('Training LSTM Regressor...')
lstm_model = train_reg_model(lstm_model, train_loader, val_loader)

lstm_model.eval()
ap, at = [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        ap.extend(np.clip(lstm_model(seq_b.to(device), X_b.to(device)).cpu().numpy(), 0, None))
        at.extend(y_b.numpy())
ap, at = np.array(ap), np.array(at)
lstm_mae = mean_absolute_error(at, ap); lstm_r2 = r2_score(at, ap)
results['LSTM'] = {'all': {'MAE': lstm_mae, 'RMSE': float(np.sqrt(mean_squared_error(at,ap))), 'R2': lstm_r2, 'MedAE': float(np.median(np.abs(at-ap)))}}
print(f'LSTM  MAE={lstm_mae:.2f}d  R2={lstm_r2:.4f}')

Training LSTM Regressor...
  Epoch 01  val_MAE=0.384d
  Epoch 02  val_MAE=0.409d
  Epoch 03  val_MAE=0.343d
  Epoch 04  val_MAE=0.322d
  Epoch 05  val_MAE=0.308d
  Epoch 06  val_MAE=0.317d
  Epoch 07  val_MAE=0.363d
  Epoch 08  val_MAE=0.312d
  Epoch 09  val_MAE=0.337d
  Epoch 10  val_MAE=0.331d
  Early stop at epoch 10
LSTM  MAE=0.12d  R2=0.0596


## 8. Transformer Regressor

In [ ]:
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div); pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))
    def forward(self, x): return self.dropout(x + self.pe[:, :x.size(1)])

class TransformerReg(nn.Module):
    def __init__(self, vocab_size, emb_dim=64, nhead=4, n_layers=2, d_ff=128, dropout=0.1, n_tab=10):
        super().__init__()
        self.emb = nn.Embedding(vocab_size, emb_dim, padding_idx=0)
        self.pe  = PositionalEncoding(emb_dim, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(d_model=emb_dim, nhead=nhead,
                    dim_feedforward=d_ff, dropout=dropout, batch_first=True)
        self.encoder = nn.TransformerEncoder(enc_layer, num_layers=n_layers)
        self.fc1 = nn.Linear(emb_dim + n_tab, 64); self.fc2 = nn.Linear(64, 1)
        self.drop = nn.Dropout(dropout)
    def forward(self, seq, tab):
        pad_mask = (seq == 0)
        x = self.pe(self.emb(seq))
        x = self.encoder(x, src_key_padding_mask=pad_mask)
        mask_f = (~pad_mask).unsqueeze(-1).float()
        x = (x * mask_f).sum(1) / mask_f.sum(1).clamp(min=1)
        return self.fc2(torch.relu(self.fc1(self.drop(torch.cat([x, tab], 1))))).squeeze(1)

tf_model = TransformerReg(VOCAB_SIZE, n_tab=N_TAB).to(device)
print('Training Transformer Regressor...')
tf_model = train_reg_model(tf_model, train_loader, val_loader)

tf_model.eval()
ap, at = [], []
with torch.no_grad():
    for X_b, seq_b, y_b in test_loader:
        ap.extend(np.clip(tf_model(seq_b.to(device), X_b.to(device)).cpu().numpy(), 0, None))
        at.extend(y_b.numpy())
ap, at = np.array(ap), np.array(at)
tf_mae = mean_absolute_error(at, ap); tf_r2 = r2_score(at, ap)
results['Transformer'] = {'all': {'MAE': tf_mae, 'RMSE': float(np.sqrt(mean_squared_error(at,ap))), 'R2': tf_r2, 'MedAE': float(np.median(np.abs(at-ap)))}}
print(f'Transformer  MAE={tf_mae:.2f}d  R2={tf_r2:.4f}')

Training Transformer Regressor...
  Epoch 01  val_MAE=0.387d
  Epoch 02  val_MAE=0.353d
  Epoch 03  val_MAE=0.356d
  Epoch 04  val_MAE=0.335d
  Epoch 05  val_MAE=0.317d
  Epoch 06  val_MAE=0.353d
  Epoch 07  val_MAE=0.335d
  Epoch 08  val_MAE=0.323d
  Epoch 09  val_MAE=0.353d
  Epoch 10  val_MAE=0.311d
  Epoch 11  val_MAE=0.334d
  Epoch 12  val_MAE=0.341d
  Epoch 13  val_MAE=0.304d
  Epoch 14  val_MAE=0.363d
  Epoch 15  val_MAE=0.346d
  Epoch 16  val_MAE=0.343d
  Epoch 17  val_MAE=0.333d
  Epoch 18  val_MAE=0.322d
  Early stop at epoch 18
Transformer  MAE=0.10d  R2=0.1437


## 9. Results Table + MAE vs Prefix Plot

In [ ]:
print('\n=== FINAL RESULTS ===')
for model, plen_dict in results.items():
    for plen, m in plen_dict.items():
        print(f'{model:22s}  prefix={plen:4s}  MAE={m["MAE"]:.2f}d  RMSE={m["RMSE"]:.2f}d  R2={m["R2"]:.4f}  MedAE={m["MedAE"]:.2f}d')

fig, ax = plt.subplots(figsize=(10, 5))
for mkey in ['Boolean/XGB','Boolean/LGBM','Succession/XGB','Succession/LGBM']:
    if mkey not in results: continue
    pls = [str(p) for p in PREFIX_LENGTHS]
    maes = [results[mkey].get(p, {}).get('MAE', np.nan) for p in pls]
    ax.plot(pls, maes, marker='o', label=mkey)
ax.set_xlabel('Prefix Length'); ax.set_ylabel('MAE (days)')
ax.set_title('Remaining Time: MAE vs Prefix Length')
ax.legend(); plt.tight_layout()
plt.savefig('figs/remaining_time_mae_vs_prefix.png', bbox_inches='tight', dpi=150)
plt.show()
print('Saved figs/remaining_time_mae_vs_prefix.png')


=== FINAL RESULTS ===
Boolean/XGB             prefix=1     MAE=20.33d  RMSE=21.77d  R2=-0.8292  MedAE=20.12d
Boolean/XGB             prefix=3     MAE=4.85d  RMSE=13.87d  R2=0.0983  MedAE=0.06d
Boolean/XGB             prefix=5     MAE=3.16d  RMSE=11.17d  R2=-0.2353  MedAE=0.06d
Boolean/XGB             prefix=10    MAE=1.26d  RMSE=6.62d  R2=-0.0304  MedAE=0.02d
Boolean/XGB             prefix=20    MAE=0.36d  RMSE=3.82d  R2=-1.6328  MedAE=0.01d
Boolean/XGB             prefix=all   MAE=0.29d  RMSE=3.46d  R2=-4.2723  MedAE=0.03d
Boolean/LGBM            prefix=1     MAE=22.20d  RMSE=23.44d  R2=-1.1210  MedAE=22.69d
Boolean/LGBM            prefix=3     MAE=7.55d  RMSE=12.13d  R2=0.3109  MedAE=4.36d
Boolean/LGBM            prefix=5     MAE=3.77d  RMSE=10.20d  R2=-0.0295  MedAE=1.00d
Boolean/LGBM            prefix=10    MAE=1.26d  RMSE=5.75d  R2=0.2210  MedAE=0.22d
Boolean/LGBM            prefix=20    MAE=0.63d  RMSE=2.39d  R2=-0.0319  MedAE=0.45d
Boolean/LGBM            prefix=all   MAE=0.46d